In [0]:
# ============================================================
# Understand the actual data distribution
# ============================================================
df = spark.sql("""
    SELECT
        COUNT(*) AS total_rows,

        -- numberDoctors breakdown
        SUM(CASE WHEN numberDoctors IS NULL THEN 1 ELSE 0 END) AS doctors_null,
        SUM(CASE WHEN CAST(numberDoctors AS STRING) = '0' THEN 1 ELSE 0 END) AS doctors_zero,
        SUM(CASE WHEN CAST(numberDoctors AS DOUBLE) > 0 THEN 1 ELSE 0 END) AS doctors_known,

        -- capacity breakdown
        SUM(CASE WHEN capacity IS NULL THEN 1 ELSE 0 END) AS capacity_null,
        SUM(CASE WHEN CAST(capacity AS STRING) = '0' THEN 1 ELSE 0 END) AS capacity_zero,
        SUM(CASE WHEN CAST(capacity AS DOUBLE) > 0 THEN 1 ELSE 0 END) AS capacity_known,

        -- procedure breakdown (Array)
        SUM(CASE WHEN procedure IS NULL THEN 1 ELSE 0 END) AS procedure_null,
        SUM(CASE WHEN size(procedure) = 0 THEN 1 ELSE 0 END) AS procedure_empty,
        SUM(CASE WHEN size(procedure) > 0 THEN 1 ELSE 0 END) AS procedure_known,

        -- equipment breakdown (Array)
        SUM(CASE WHEN equipment IS NULL THEN 1 ELSE 0 END) AS equipment_null,
        SUM(CASE WHEN size(equipment) = 0 THEN 1 ELSE 0 END) AS equipment_empty,
        SUM(CASE WHEN size(equipment) > 0 THEN 1 ELSE 0 END) AS equipment_known,

        -- specialties breakdown (Array)
        SUM(CASE WHEN specialties IS NULL THEN 1 ELSE 0 END) AS specialties_null,
        SUM(CASE WHEN size(specialties) = 0 THEN 1 ELSE 0 END) AS specialties_empty,
        SUM(CASE WHEN size(specialties) > 0 THEN 1 ELSE 0 END) AS specialties_known,

        -- capability breakdown (Array)
        SUM(CASE WHEN capability_cleaned IS NULL THEN 1 ELSE 0 END) AS capability_null,
        SUM(CASE WHEN size(capability_cleaned) = 0 THEN 1 ELSE 0 END) AS capability_empty,
        SUM(CASE WHEN size(capability_cleaned) > 0 THEN 1 ELSE 0 END) AS capability_known

    FROM workspace.default.facilities3
""")

display(df)

In [0]:
# ============================================================
# Cell 1 — Load table
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, col, when, size
from pyspark.sql.types import IntegerType, FloatType

df = spark.read.table("workspace.default.facilities3")

In [0]:
# ============================================================
# Cell 2 — Count columns using size() on true arrays
# ============================================================
df = df \
    .withColumn("procedure_count",
        when(col("procedure").isNull(), 0)
        .otherwise(F.size(col("procedure")))) \
    .withColumn("equipment_count",
        when(col("equipment").isNull(), 0)
        .otherwise(F.size(col("equipment")))) \
    .withColumn("specialty_count",
        when(col("specialties").isNull(), 0)
        .otherwise(F.size(col("specialties")))) \
    .withColumn("capability_count",
        when(col("capability_cleaned").isNull(), 0)
        .otherwise(F.size(col("capability_cleaned"))))

df.select(
    F.avg("specialty_count").alias("avg_specialties"),
    F.avg("capability_count").alias("avg_capability"),
    F.avg("procedure_count").alias("avg_procedures"),
    F.avg("equipment_count").alias("avg_equipment"),
    F.max("specialty_count").alias("max_specialties"),
    F.max("capability_count").alias("max_capability"),
).display()

In [0]:
# ============================================================
# Cell 3 — Binary claim flags
# ============================================================
SURGICAL_KW  = [
    "surg", "operation", "theatre", "theater", "caesarean", "cesarean", 
    "c-section", "c section", "laparoscop", "endoscop", "anaesth", 
    "transplant", "amputation", "appendectomy", "hernia"
]

IMAGING_KW   = [
    "mri", "ct scan", "x-ray", "xray", "ultrasound", "radiology", 
    "imaging", "mammograph", "scan", "sonograph", "echocardiograph", 
    "fluoroscopy", "radiograph"
]

EMERGENCY_KW = [
    "emergency", "casualty", "trauma", "24-hour emergency", 
    "24/7 emergency", "accident and emergency", "a&e", "urgent care", 
    "resuscitat", "ambulance"
]

ICU_KW       = [
    "icu", "intensive care", "critical care", "nicu", "picu", 
    "high dependency", "hdu", "ccu", "ventilator"
]

MATERNITY_KW = [
    "maternity", "obstetric", "labour", "labor", "delivery", 
    "antenatal", "postnatal", "anc", "pnc", "midwife", "midwifery", 
    "gynaecol", "gynecol", "cwc"
]
def array_to_text(arr):
    if arr is None:
        return ""
    if isinstance(arr, list):
        return " ".join([str(x) for x in arr if x]).lower()
    return str(arr).lower()

def make_flag_udf(keywords):
    def flag(cap_cleaned, proc):
        # Use capability_cleaned (not raw capability) for cleaner signals
        combined = array_to_text(cap_cleaned) + " " + array_to_text(proc)
        return int(any(kw in combined for kw in keywords))
    return udf(flag, IntegerType())

has_surgical_udf  = make_flag_udf(SURGICAL_KW)
has_imaging_udf   = make_flag_udf(IMAGING_KW)
has_emergency_udf = make_flag_udf(EMERGENCY_KW)
has_icu_udf       = make_flag_udf(ICU_KW)
has_maternity_udf = make_flag_udf(MATERNITY_KW)

# Imaging also checks equipment (where imaging equipment would be listed)
def has_imaging_flag(cap_cleaned, equipment, proc):
    combined = (array_to_text(cap_cleaned) + " " +
                array_to_text(equipment)   + " " +
                array_to_text(proc))
    return int(any(kw in combined for kw in IMAGING_KW))

has_imaging_full_udf = udf(has_imaging_flag, IntegerType())

df = df \
    .withColumn("has_surgical_claim",
        has_surgical_udf(col("capability_cleaned"), col("procedure"))) \
    .withColumn("has_imaging_claim",
        has_imaging_full_udf(col("capability_cleaned"), col("equipment"), col("procedure"))) \
    .withColumn("has_emergency_claim",
        has_emergency_udf(col("capability_cleaned"), col("procedure"))) \
    .withColumn("has_icu_claim",
        has_icu_udf(col("capability_cleaned"), col("procedure"))) \
    .withColumn("has_maternity_claim",
        has_maternity_udf(col("capability_cleaned"), col("procedure")))

# Verify flag distribution
df.select(
    F.sum("has_surgical_claim").alias("surgical"),
    F.sum("has_imaging_claim").alias("imaging"),
    F.sum("has_emergency_claim").alias("emergency"),
    F.sum("has_icu_claim").alias("icu"),
    F.sum("has_maternity_claim").alias("maternity"),
).display()

In [0]:
# ============================================================
# Cell 4 — Alignment UDFs
# Adjusted for 51% capability fill rate
# ============================================================
SPECIALTY_TO_KEYWORDS = {
    "cardiology"             : ["cardiac", "cardio", "heart", "ecg", "echocardiograph"],
    "generalSurgery"         : ["surg", "operation", "theatre", "theater", "anaesth", "appendectomy", "hernia"],
    "orthopedicSurgery"      : ["orthop", "fracture", "bone", "joint", "implant", "trauma"],
    "gynecologyAndObstetrics": ["obstetric", "maternity", "labour", "labor",
                                "delivery", "antenatal", "postnatal", "gynaecol", "gynecol", "anc", "pnc", "c-section", "cesarean"],
    "pediatrics"             : ["paediatric", "pediatric", "neonatal", "child health", "nicu", "picu", "cwc"],
    "ophthalmology"          : ["eye", "ophthalm", "vision", "cataract", "retina", "optometr", "glaucoma"],
    "dentistry"              : ["dental", "dentist", "tooth", "teeth", "oral", "maxillofacial"],
    "emergencyMedicine"      : ["emergency", "casualty", "trauma", "resuscitat", "accident", "a&e", "ambulance"],
    "internalMedicine"       : ["internal medicine", "general medicine", "physician", "outpatient", "diabetes", "endocrin"],
    "psychiatry"             : ["psychiatr", "mental health", "psycholog", "counseling"],
    "nephrology"             : ["kidney", "renal", "dialysis", "nephro"],
    "oncology"               : ["cancer", "oncol", "chemotherapy", "radiotherapy"],
    "neurology"              : ["neuro", "brain", "spine", "seizure", "stroke", "eeg"],
    "urology"                : ["urol", "bladder", "prostate"],
    "dermatology"            : ["dermatol", "skin"],
    "radiology"              : ["radiol", "imaging", "mri", "ct scan", "x-ray",
                                "xray", "ultrasound", "scan", "sonograph", "fluoroscopy"],
    "familyMedicine"         : ["family medicine", "general practice", "primary care", "family health"],
}

def specialty_capability_alignment(specialties_val, capability_val, procedure_val):
    """
    Counts specialties with no supporting evidence.

    Key rule: if capability_cleaned AND procedure are BOTH empty,
    return 0 — we cannot do alignment check, not suspicious.
    This handles the 40% of rows where capability is empty after cleaning.
    """
    spec_list = specialties_val if isinstance(specialties_val, list) else []
    if not spec_list:
        return 0

    cap_text  = array_to_text(capability_val)
    proc_text = array_to_text(procedure_val)

    # Cannot check alignment without any supporting text — not suspicious
    if not cap_text.strip() and not proc_text.strip():
        return 0

    support_text = (cap_text + " " + proc_text).strip()
    unsupported  = 0

    for spec in spec_list:
        spec_clean = str(spec).strip()
        keywords   = SPECIALTY_TO_KEYWORDS.get(spec_clean, [])
        if not keywords:
            continue
        if not any(kw in support_text for kw in keywords):
            unsupported += 1

    return unsupported

alignment_udf = udf(specialty_capability_alignment, IntegerType())


def capability_without_specialty(capability_val, specialties_val):
    """
    Counts high-stakes capability claims with no matching specialty.
    Uses capability_cleaned so this is now meaningful.
    """
    cap_list  = capability_val  if isinstance(capability_val,  list) else []
    spec_list = specialties_val if isinstance(specialties_val, list) else []

    if not cap_list:
        return 0

    cap_text  = array_to_text(cap_list)
    spec_text = array_to_text(spec_list)

    HIGH_STAKES = {
            "cardiac"  : ["cardiology"],   
            "heart"    : ["cardiology"],
            "dialysis" : ["nephrology", "internalmedicine"], 
            "renal"    : ["nephrology", "internalmedicine"],
            "cancer"   : ["oncology"],    
            "oncol"    : ["oncology", "chemotherapy"],
            "neuro"    : ["neurology", "neurosurgery"],   
            "stroke"   : ["neurology", "internalmedicine"],
            "psychiatr": ["psychiatry"],  
            "mental"   : ["psychiatry"],
            "mri"      : ["radiology"],
            "ct scan"  : ["radiology"],
            
            # Surgical terms map to ANY surgical specialty
            "surg"     : ["generalsurgery", "orthopedicsurgery", "neurosurgery", "pediatricsurgery", "plasticsurgery", "gynecologyandobstetrics"],
            "theatre"  : ["generalsurgery", "orthopedicsurgery", "neurosurgery", "pediatricsurgery", "plasticsurgery", "gynecologyandobstetrics"],
            
            # Maternity terms
            "obstetric": ["gynecologyandobstetrics"],
            "maternity": ["gynecologyandobstetrics", "familymedicine", "generalpractice"],
            "c-section": ["gynecologyandobstetrics", "generalsurgery"],
            
            # Peds terms
            "paediatric": ["pediatrics", "pediatricsurgery"], 
            "nicu"      : ["pediatrics", "gynecologyandobstetrics"]
        }
    
    orphaned = set()

    for kw, acceptable_specs in HIGH_STAKES.items():
        if kw in cap_text:
            # Check if AT LEAST ONE acceptable specialty is in the facility's specialty list
            if not any(acc_spec in spec_text for acc_spec in acceptable_specs):
                # If missing all of them, flag the primary missing specialty
                orphaned.add(acceptable_specs[0]) 
                    
    return len(orphaned)
cap_without_spec_udf = udf(capability_without_specialty, IntegerType())

df = df \
    .withColumn("unsupported_specialty_count",
        alignment_udf(
            col("specialties"),
            col("capability_cleaned"),
            col("procedure")
        )) \
    .withColumn("capability_without_specialty_count",
        cap_without_spec_udf(
            col("capability_cleaned"),
            col("specialties")
        ))

In [0]:
# ============================================================
# Cell 5 — Anomaly scoring adjusted for your distribution
# ============================================================

def claim_inflation_score(spec_count, cap_count, proc_count, equip_count,
                          unsupported_specs, cap_without_spec):
    sc  = int(spec_count        or 0)
    cc  = int(cap_count         or 0)
    pc  = int(proc_count        or 0)
    ec  = int(equip_count       or 0)
    us  = int(unsupported_specs  or 0)
    cws = int(cap_without_spec   or 0)
    score = 0

    # ── Signal 1: specialty-capability misalignment ───────────
    # Only fires when capability_cleaned has content (51% of rows)
    # us=0 when capability is empty — so no false positives
    if us >= 3: score += 40
    elif us == 2: score += 25
    elif us == 1: score += 12

    # ── Signal 2: capability claims without specialty ─────────
    # capability_cleaned is now clean so this is a reliable signal
    if cws >= 3: score += 35
    elif cws == 2: score += 20
    elif cws == 1: score += 10

    # ── Signal 3: high specialty count, empty capability ──────
    # With 51% fill rate, empty capability is meaningful
    # (but less so than before cleaning — some facilities just
    #  had no real capabilities to extract)
    # Use higher threshold to reduce false positives
    if sc >= 5 and cc == 0: score += 30
    elif sc >= 4 and cc == 0: score += 20
    elif sc >= 3 and cc == 0: score += 10

    # ── Signal 4: procedure/equipment ratio ───────────────────
    # Only fires for 87 facilities — but when it does, it's reliable
    if pc > 0 and ec > 0:
        ratio = pc / ec
        if ratio > 8: score += 20
        elif ratio > 5: score += 10

    # ── Signal 5: procedures with zero equipment ──────────────
    # 178 facilities have procedures — when present, no equipment is a flag
    if pc > 5 and ec == 0: score += 15
    elif pc > 2 and ec == 0: score += 8

    return min(score, 100)


def infrastructure_mismatch_score(has_surgical, has_imaging, has_icu,
                                   has_emergency, equip_count,
                                   cap_count):
    """
    Uses equipment as primary infrastructure signal.
    Uses capability_count (cleaned) as secondary signal.
    """
    ec  = int(equip_count  or 0)
    cc  = int(cap_count    or 0)
    hs  = int(has_surgical or 0)
    hi  = int(has_imaging  or 0)
    hu  = int(has_icu      or 0)
    he  = int(has_emergency or 0)
    score = 0

    # Surgery + no equipment
    # (surgical flags came from capability_cleaned — so this is reliable)
    if hs and ec == 0: score += 40
    elif hs and ec == 1: score += 20

    # Imaging + no equipment (strongest signal — MRI needs physical hardware)
    if hi and ec == 0: score += 45
    elif hi and ec == 1: score += 20

    # ICU + empty capability (after cleaning, empty cap = genuinely sparse)
    if hu and cc == 0: score += 35
    elif hu and cc <= 1: score += 15

    # Emergency + empty capability
    if he and cc == 0: score += 20

    return min(score, 100)


def overall_anomaly_score(cs, ms):
    return min(round(int(cs or 0) * 0.4 + int(ms or 0) * 0.6), 100)


def data_completeness(sc, cc, pc, ec, nd, cap, desc):
    """
    Scoring weights reflect YOUR actual distribution:
    specialties (91%) and capability_cleaned (51%) are most reliable.
    """
    score = 0
    if int(sc  or 0) > 0: score += 2   # specialties — most reliable (91%)
    if int(cc  or 0) > 0: score += 2   # capability_cleaned — reliable (51%)
    if int(pc  or 0) > 0: score += 1   # procedure — sparse (22%)
    if int(ec  or 0) > 0: score += 1   # equipment — very sparse (11%)
    if int(nd  or 0) > 0: score += 1   # doctors — almost always unknown
    if int(cap or 0) > 0: score += 1   # bed capacity — almost always unknown
    if desc and str(desc).strip() not in ("", "None", "null"): score += 1
    return min(score, 6)


claim_score_udf    = udf(claim_inflation_score,         IntegerType())
mismatch_score_udf = udf(infrastructure_mismatch_score, IntegerType())
anomaly_score_udf  = udf(overall_anomaly_score,         IntegerType())
completeness_udf   = udf(data_completeness,             IntegerType())

df = df \
    .withColumn("num_doctors_clean",
        when(col("numberDoctors").isNull(), 0)
        .otherwise(col("numberDoctors").cast("int"))) \
    .withColumn("capacity_clean",
        when(col("capacity").isNull(), 0)
        .otherwise(col("capacity").cast("int"))) \
    .withColumn("proc_equip_ratio",
        when(
            (col("procedure_count") > 0) & (col("equipment_count") > 0),
            (col("procedure_count") / col("equipment_count")).cast(FloatType())
        ).otherwise(None)) \
    .withColumn("claim_inflation_score",
        claim_score_udf(
            col("specialty_count"), col("capability_count"),
            col("procedure_count"), col("equipment_count"),
            col("unsupported_specialty_count"),
            col("capability_without_specialty_count")
        )) \
    .withColumn("infrastructure_mismatch_score",
        mismatch_score_udf(
            col("has_surgical_claim"), col("has_imaging_claim"),
            col("has_icu_claim"),      col("has_emergency_claim"),
            col("equipment_count"),   col("capability_count")
        )) \
    .withColumn("anomaly_score",
        anomaly_score_udf(
            col("claim_inflation_score"),
            col("infrastructure_mismatch_score")
        )) \
    .withColumn("data_completeness_score",
        completeness_udf(
            col("specialty_count"),   col("capability_count"),
            col("procedure_count"),   col("equipment_count"),
            col("num_doctors_clean"), col("capacity_clean"),
            col("description")
        ))

In [0]:
# ============================================================
# Cell 6 — Write enriched table
# ============================================================
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.facilities3")

print("Enriched table written")

In [0]:
# ============================================================
# Cell 7 — Validate score distribution
# ============================================================
result_df = spark.sql("""
    SELECT
        CASE
            WHEN anomaly_score >= 60 THEN '60-100 High'
            WHEN anomaly_score >= 35 THEN '35-59 Medium'
            WHEN anomaly_score >= 15 THEN '15-34 Low'
            ELSE '0-14 Clean'
        END AS band,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM workspace.default.facilities3
    GROUP BY 1
    ORDER BY band DESC
""")

display(result_df)

In [0]:
# ============================================================
# Cell 8 — Top anomalies with readable explanation
# ============================================================
spark.sql("""
    SELECT
        name,
        facilityTypeId,
        address_city,
        address_stateOrRegion,
        specialty_count,
        capability_count,
        procedure_count,
        equipment_count,
        unsupported_specialty_count,
        capability_without_specialty_count,
        has_surgical_claim,
        has_imaging_claim,
        has_icu_claim,
        has_emergency_claim,
        claim_inflation_score,
        infrastructure_mismatch_score,
        anomaly_score,
        data_completeness_score
    FROM workspace.default.facilities3
    WHERE anomaly_score >= 35
    ORDER BY anomaly_score DESC
    LIMIT 20
""").display()

In [0]:
# ============================================================
# Cell 9 — The 249 facilities that lost all capability data
# These are themselves an interesting finding for the hackathon
# ============================================================
spark.sql("""
    SELECT
        name,
        facilityTypeId,
        address_city,
        address_stateOrRegion,
        specialty_count,
        anomaly_score,
        data_completeness_score
    FROM workspace.default.facilities3
    WHERE size(capability_cleaned) = 0
      AND size(capability)         > 0
      AND specialty_count          > 0
    ORDER BY specialty_count DESC
    LIMIT 20
""").display()